# Join batch benchmark — batch size sweep

Plots results from `join_batch_benchmark_sf<SF>.csv` produced by
`run_join_batch_benchmarks.sh` (default batch sizes: 64 MiB … 4 GiB, doubling).

Timings are libcudf hash build + probe only (Parquet read is excluded).

**Dependencies:** `pip install -r test/tpch_performance/notebook-requirements.txt` (or any env with `pandas` and `matplotlib`). Open the notebook from the **repo root** or from `test/tpch_performance/` (CSV path auto-detected).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

SF = 10
_here = Path.cwd()
_csv_name = f"join_batch_benchmark_sf{SF}.csv"
# Kernel cwd is usually repo root or test/tpch_performance/
_candidates = [
    _here / "test" / "tpch_performance" / _csv_name,
    _here / _csv_name,
]
CSV_PATH = next((p for p in _candidates if p.is_file()), _candidates[0])

_style = "seaborn-v0_8-whitegrid"
plt.style.use(_style if _style in plt.style.available else "default")

In [ ]:
df = pd.read_csv(CSV_PATH)
df["batch_mib"] = df["batch_size_bytes"] / (1024**2)
df["batch_gib"] = df["batch_size_bytes"] / (1024**3)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

for name, g in df.groupby("join_name", sort=True):
    g = g.sort_values("batch_mib")
    axes[0, 0].plot(g["batch_mib"], g["probe_time_ms"], marker="o", label=name)
    axes[0, 1].plot(g["batch_mib"], g["build_time_ms"], marker="o", label=name)
    axes[1, 0].plot(g["batch_mib"], g["total_time_ms"], marker="o", label=name)
    thr = g["throughput_probe_rows_per_sec"] / 1e6
    axes[1, 1].plot(g["batch_mib"], thr, marker="o", label=name)

for ax in axes.flat:
    ax.set_xscale("log", base=2)
    ax.set_xlabel("Target batch size (MiB, log₂)")
    ax.legend(fontsize=7, loc="best")

axes[0, 0].set_ylabel("Probe time (ms)")
axes[0, 0].set_title("libcudf inner_join (probe side)")
axes[0, 1].set_ylabel("Build time (ms)")
axes[0, 1].set_title("libcudf hash_join build")
axes[1, 0].set_ylabel("Total (ms)")
axes[1, 0].set_title("Build + probe")
axes[1, 1].set_ylabel("Probe throughput (M rows/s)")
axes[1, 1].set_title("probe_rows / probe_time")

fig.suptitle(f"join_batch_benchmark SF{SF} — {CSV_PATH.name}", fontsize=11)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
pivot = df.pivot_table(
    index="join_name",
    columns="batch_mib",
    values="total_time_ms",
    aggfunc="first",
)
pivot = pivot.reindex(sorted(pivot.index))
pivot = pivot[sorted(pivot.columns, key=float)]
im = ax.imshow(pivot.values, aspect="auto", cmap="viridis_r")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c:.0f}" for c in pivot.columns], rotation=45, ha="right")
ax.set_xlabel("Batch size (MiB)")
ax.set_title("Total time (ms) heatmap")
fig.colorbar(im, ax=ax, label="ms")
plt.show()